# Day 1 Lab B — From raw EEG files to a training pipeline

In the first notebook you looked at *one* seizure. Real projects spend most of their time **before** the first model is
trained: understanding the recordings, cleaning them, cutting them into training examples and splitting them so that the
evaluation is honest. This notebook builds such a pipeline for the two EEG datasets of the course, step by step.
Every step has a reason — the explanations are part of the content, read them.

**Learning goals**
1. Understand how **CHB-MIT** (PhysioNet) and **SeizeIT2** (OpenNeuro, BIDS format) are organised and compute basic statistics.
2. Look at EEG in the time and the frequency domain; recognise line noise, artifacts and seizure activity.
3. Build a preprocessing pipeline: channel selection, filtering, resampling, windowing, labelling, artifact checks, normalisation.
4. Split by **recording** instead of by window — and understand why.
5. Wrap everything in a PyTorch `Dataset` / `DataLoader`.
6. Train a quick sanity baseline (band power + logistic regression) and evaluate it with seizure-detection metrics.

**Compute:** a CPU runtime is enough. Download: about 340 MB in total (cached in `./data` while the runtime lives).
Exercises are marked with `# TODO`.

| Part | Topic | Time |
|---|---|---|
| 1 | CHB-MIT: files, channels, statistics, spectrum | 25 min |
| 2 | SeizeIT2: BIDS, annotations, wearable modalities | 20 min |
| 3 | Preprocessing: filter, resample, windows, labels, artifacts | 30 min |
| 4 | Splits and class imbalance | 10 min |
| 5 | PyTorch `Dataset` and `DataLoader` | 15 min |
| 6 | Sanity baseline and seizure-detection metrics | 20 min |

In [ ]:
!pip install -q mne

In [ ]:
import re
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.signal import welch, find_peaks
"""Small loaders for the two open-access EEG seizure datasets used in the labs.

CHB-MIT  (Shoeb 2009, PhysioNet): paediatric scalp EEG, 23 bipolar channels, 256 Hz, 1 h records.
SeizeIT2 (Bhagubai et al. 2025, OpenNeuro ds005873): wearable behind-the-ear EEG, 2 channels, 256 Hz.
Files are downloaded once and cached in ./data (in Colab: lost when the runtime is recycled).
"""
import re
import urllib.request
from pathlib import Path

import numpy as np

DATA = Path("data")
CHBMIT = "https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"  # AWS mirror of PhysioNet, much faster
SEIZEIT2 = "https://s3.amazonaws.com/openneuro.org/ds005873/"
# the first 18 CHB-MIT channels form the longitudinal bipolar "double banana" montage
CHANNELS = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1", "FP2-F4",
            "F4-C4", "C4-P4", "P4-O2", "FP2-F8", "F8-T8", "T8-P8", "P8-O2", "FZ-CZ", "CZ-PZ"]


def fetch(url, name):
    """Download url to data/name once; return the local path."""
    out = DATA / name
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", url)
        urllib.request.urlretrieve(url, out.with_suffix(".part"))
        out.with_suffix(".part").rename(out)  # no half-written files in the cache
    return out


def preprocess(raw, picks, fs):
    """Band-pass 0.5-40 Hz, resample to fs, return (channels, time) in microvolts."""
    raw.pick(picks).filter(0.5, 40, verbose="error").resample(fs, verbose="error")
    return raw.get_data() * 1e6


def windows(data, fs, win_s, events, around_s=None):
    """Cut (channels, time) into non-overlapping windows of win_s seconds.

    y = 1 if the window centre lies inside a seizure event (onset_s, end_s).
    around_s: keep only windows within around_s seconds of a seizure (None = keep all).
    """
    n = int(win_s * fs)
    k = data.shape[1] // n
    X = data[:, :k * n].reshape(data.shape[0], k, n).transpose(1, 0, 2)
    mid = (np.arange(k) + 0.5) * win_s
    y = np.zeros(k, dtype=np.int64)
    keep = np.zeros(k, dtype=bool) if around_s is not None else np.ones(k, dtype=bool)
    for a, b in events:
        y[(mid >= a) & (mid < b)] = 1
        if around_s is not None:
            keep |= (mid >= a - around_s) & (mid < b + around_s)
    return np.ascontiguousarray(X[keep], dtype=np.float32), y[keep]


def chbmit_seizures(patient="chb01"):
    """{'chb01_03': [(2996, 3036)], ...} parsed from the PhysioNet summary file."""
    text = fetch(f"{CHBMIT}{patient}/{patient}-summary.txt", f"chbmit/{patient}-summary.txt").read_text()
    out = {}
    for block in text.split("File Name: ")[1:]:
        t = [int(s) for s in re.findall(r"Seizure.*(?:Start|End) Time:\s*(\d+)", block)]
        out[block.split(".edf")[0]] = list(zip(t[::2], t[1::2]))
    return out


def load_chbmit(records=("chb01_03", "chb01_04", "chb01_15", "chb01_16", "chb01_18", "chb01_26"),
                win_s=4.0, fs=128):
    """CHB-MIT records -> X (n, 18, win_s*fs) in uV, y (n,) 1 = seizure, rec (n,) index into records."""
    import mne
    X, y, rec = [], [], []
    for i, r in enumerate(records):
        patient = r.split("_")[0]
        raw = mne.io.read_raw_edf(fetch(f"{CHBMIT}{patient}/{r}.edf", f"chbmit/{r}.edf"), preload=True, verbose="error")
        raw.rename_channels(lambda c: c.removesuffix("-0"))  # MNE renames the duplicated T8-P8 channel
        Xi, yi = windows(preprocess(raw, CHANNELS, fs), fs, win_s, chbmit_seizures(patient)[r])
        X.append(Xi), y.append(yi), rec.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(rec)


def seizeit2_events(subject="sub-001"):
    """{run_path: [(onset_s, end_s), ...]} seizure events of every EEG run of one SeizeIT2 subject."""
    listing = urllib.request.urlopen(
        f"https://s3.amazonaws.com/openneuro.org?list-type=2&prefix=ds005873/{subject}/").read().decode()
    out = {}
    for stem in re.findall(r"<Key>ds005873/([^<]+?/eeg/[^<]+?)_events\.tsv</Key>", listing):
        rows = fetch(f"{SEIZEIT2}{stem}_events.tsv", f"seizeit2/{stem}_events.tsv").read_text().splitlines()[1:]
        cols = [r.split("\t") for r in rows]
        out[stem] = [(float(c[0]), float(c[0]) + float(c[1])) for c in cols if c[2].startswith("sz")]
    return out


def load_seizeit2(subject="sub-001", win_s=4.0, fs=128, around_s=1800):
    """Seizure-containing runs of one SeizeIT2 subject -> X (n, 2, win_s*fs) in uV, y (n,), run (n,).

    around_s keeps +-30 min around each seizure, otherwise one subject is >100 h of mostly background.
    """
    import mne
    X, y, run = [], [], []
    for i, (stem, events) in enumerate(e for e in seizeit2_events(subject).items() if e[1]):
        raw = mne.io.read_raw_edf(fetch(f"{SEIZEIT2}{stem}_eeg.edf", f"seizeit2/{stem}_eeg.edf"), preload=True, verbose="error")
        Xi, yi = windows(preprocess(raw, raw.ch_names[:2], fs), fs, win_s, events, around_s)
        X.append(Xi), y.append(yi), run.append(np.full(len(yi), i))
    return np.concatenate(X), np.concatenate(y), np.concatenate(run)


def band_power(X, fs=128):
    """Log band power (delta, theta, alpha, beta, low gamma) per channel: (n, ch, t) -> (n, ch*5)."""
    from scipy.signal import welch
    f, p = welch(X, fs=fs, nperseg=fs, axis=-1)
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 40)]
    bp = np.stack([p[..., (f >= a) & (f < b)].mean(-1) for a, b in bands], -1)
    return np.log(bp + 1e-10).reshape(len(X), -1).astype(np.float32)
import mne

np.random.seed(0)
torch.manual_seed(0)

## Part 1 — CHB-MIT: what is in the dataset?
CHB-MIT contains scalp EEG of 22 children with epilepsy, recorded at Boston Children's Hospital (23 cases, one patient was
recorded twice). Each patient has a folder (`chb01/`, `chb02/`, …) with **one-hour EDF files** (`chb01_01.edf`, …) and a
**summary file** that lists the channels and the seizure times. EDF ("European Data Format") is the standard file format
for biosignals: a header (channel names, sampling rate, units) followed by the samples.

The summary file of patient `chb01` is a small text file — let us read it.

In [ ]:
summary = fetch(CHBMIT + "chb01/chb01-summary.txt", "chbmit/chb01-summary.txt").read_text()
print(summary[:1700])

What you see:
- **Sampling rate** 256 Hz: 256 values per second and channel.
- **Channels** are *bipolar*: `FP1-F7` is the voltage difference between electrodes FP1 and F7. Channels 1–18 form the
  standard "double banana" montage (see the slide *EEG in one slide*); channels 19–23 are extra.
- Per file: clock start/end time, number of seizures and, for each seizure, start/end **in seconds from the file start**.

The helper `chbmit_seizures()` (from the course module) parses the seizure times. The function below adds the duration of
every file from the clock times.

In [ ]:
def file_durations(summary):
    """{'chb01_01': 3600, ...}: duration of every file in seconds, from the clock times in the summary."""
    out = {}
    for block in summary.split("File Name: ")[1:]:
        name = block.split(".edf")[0]
        t = re.findall(r"File (?:Start|End) Time:\s*(\d+):(\d+):(\d+)", block)
        (h0, m0, s0), (h1, m1, s1) = [tuple(map(int, x)) for x in t]
        out[name] = ((h1 - h0) * 3600 + (m1 - m0) * 60 + (s1 - s0)) % 86400  # recordings may pass midnight
    return out

durations = file_durations(summary)
seizures = chbmit_seizures("chb01")
print(list(seizures.items())[:4])

**Exercise 1.1 — dataset statistics.** Using `durations` and `seizures`, compute for patient chb01:
`n_files`, `total_hours`, `n_seizures`, the list `sz_durations` (seconds per seizure) and `sz_fraction` (fraction of the
recorded time that is seizure). Then answer: what does `sz_fraction` mean for training a classifier?

In [ ]:
# TODO: n_files, total_hours, n_seizures, sz_durations, sz_fraction
pass  # YOUR CODE HERE
print(f"{n_files} files, {total_hours:.1f} h, {n_seizures} seizures, "
      f"durations {min(sz_durations)}-{max(sz_durations)} s (mean {np.mean(sz_durations):.0f} s), "
      f"seizure fraction {100 * sz_fraction:.2f} %")
assert n_files == 42 and n_seizures == 7 and abs(total_hours - 40.55) < 0.05

*Your answer:*

### 1.2 One recording
`chb01_03` contains the seizure you saw in the first notebook. MNE reads EDF files and converts all values to **volts**.

In [ ]:
raw = mne.io.read_raw_edf(fetch(CHBMIT + "chb01/chb01_03.edf", "chbmit/chb01_03.edf"), preload=True, verbose="error")
print(raw)
print("sampling rate:", raw.info["sfreq"], "Hz | duration:", raw.n_times / raw.info["sfreq"] / 60, "min")
print(raw.ch_names)

Two details that trip up many projects:
- `T8-P8` exists twice in the file. MNE renames the duplicates to `T8-P8-0` and `T8-P8-1`.
- Channel 19 `P7-T7` looks suspiciously like channel 3 `T7-P7`.

**Exercise 1.2.** Compute the correlation coefficient (`np.corrcoef`) between `T7-P7` and `P7-T7` (`r_inverted`) and between
`T8-P8-0` and `T8-P8-1` (`r_duplicate`). Use `raw.get_data(picks=[name])[0]`. What do you conclude for the channel selection?

In [ ]:
# TODO: r_inverted, r_duplicate
pass  # YOUR CODE HERE
print(f"r(T7-P7, P7-T7) = {r_inverted:.4f}   r(T8-P8-0, T8-P8-1) = {r_duplicate:.4f}")
assert r_inverted < -0.99 and r_duplicate > 0.99

*Your answer:*

In [ ]:
raw.rename_channels(lambda c: c.removesuffix("-0"))  # T8-P8-0 -> T8-P8, so that CHANNELS can be picked by name
onset, end = seizures["chb01_03"][0]

### 1.3 Time domain and frequency domain
The **power spectral density (PSD)** shows how much power a signal has at each frequency. We estimate it with Welch's
method: cut the signal into short overlapping pieces, take the FFT of each, average the squared magnitudes.

In [ ]:
fs_raw = raw.info["sfreq"]
seg_bg = raw.get_data(picks=["F7-T7"], tmin=onset - 300, tmax=onset - 260)[0] * 1e6  # 40 s background, in uV
seg_sz = raw.get_data(picks=["F7-T7"], tmin=onset, tmax=end)[0] * 1e6               # the 40 s seizure

t = np.arange(2560) / fs_raw  # first 10 s of both segments
plt.figure(figsize=(14, 3.5))
plt.plot(t, seg_bg[:2560], lw=0.6, label="background")
plt.plot(t, seg_sz[:2560] - 400, lw=0.6, label="seizure (shifted by -400 uV)")
plt.xlabel("time [s]"); plt.ylabel("uV"); plt.legend(); plt.title("F7-T7, first 10 s"); plt.show()

**Exercise 1.3.** Estimate the PSD of `seg_bg` and `seg_sz` with `f, p = welch(x, fs=fs_raw, nperseg=1024)` and plot both on a
logarithmic y-axis (`plt.semilogy`). Store the background PSD in `f_bg, p_bg`.
1. There is a sharp peak that has nothing to do with the brain. At which frequency, and where does it come from?
2. What changes during the seizure?

In [ ]:
# TODO: f_bg, p_bg (background) and the seizure PSD; plot both
pass  # YOUR CODE HERE
peak = p_bg[(f_bg > 59.5) & (f_bg < 60.5)].mean() / p_bg[(f_bg > 55) & (f_bg < 58)].mean()
print(f"power at 60 Hz is {peak:.0f}x the power at 55-58 Hz")

*Your answer:*

## Part 2 — SeizeIT2: a wearable dataset in BIDS format
SeizeIT2 (Bhagubai et al., 2025) was recorded with a small wearable device in five European epilepsy monitoring units:
**two EEG channels from electrodes behind the ears** plus ECG, EMG and movement sensors. It follows the **BIDS** standard
(Brain Imaging Data Structure), which you will meet in many open neuro datasets:

```
ds005873/sub-001/ses-01/eeg/sub-001_ses-01_task-szMonitoring_run-07_eeg.edf      <- signals
                            eeg/..._run-07_eeg.json                              <- metadata (sampling rate, montage, ...)
                            eeg/..._run-07_events.tsv                            <- annotations (seizures, ...)
                            ecg/..._run-07_ecg.edf   emg/..._run-07_emg.edf   mov/..._run-07_mov.edf
```
One *run* is one continuous recording (several hours). The listing of subject `sub-001`, run 07:

In [ ]:
listing = urllib.request.urlopen(
    "https://s3.amazonaws.com/openneuro.org?list-type=2&prefix=ds005873/sub-001/").read().decode()
for key, size in re.findall(r"<Key>ds005873/([^<]+)</Key>.*?<Size>(\d+)</Size>", listing):
    if "run-07" in key:
        print(f"{int(size) / 1e6:6.1f} MB  {key}")

The annotations are a tab-separated table. `eventType` codes the seizure type following the ILAE 2017 classification:
`sz_foc_ia_m_tonic` = focal seizure, impaired awareness, motor onset, tonic. `bckg` = background (no seizure),
`impd` = impedance check of the electrodes (the data are probably artifacts there).

In [ ]:
runs = sorted(seizeit2_events("sub-001"))  # downloads the 9 small events.tsv files of sub-001
events07 = pd.read_csv(f"data/seizeit2/{runs[6]}_events.tsv", sep="\t")
print(runs[6])
events07

**Exercise 2.1 — statistics of one subject.** Read the `events.tsv` of all 9 runs of `sub-001`
(`pd.read_csv(f"data/seizeit2/{run}_events.tsv", sep="\t")`). Compute `total_hours` (column `recordingDuration`, same value in
every row of a run), `n_seizures` (rows whose `eventType` starts with `"sz"`) and `n_impd` (impedance checks).
Compare with CHB-MIT: hours per seizure?

In [ ]:
# TODO: total_hours_s2, n_seizures_s2, n_impd
pass  # YOUR CODE HERE
print(f"sub-001: {len(runs)} runs, {total_hours_s2:.1f} h, {n_seizures_s2} seizures, {n_impd} impedance checks")
assert n_seizures_s2 == 4 and abs(total_hours_s2 - 106.6) < 0.2

*Your answer:*

### 2.2 All modalities around a seizure
Run 07 contains a 143-s focal seizure. The cell below loads all four modalities (≈ 90 MB) but reads only ±2 min around the
seizure into memory. Note the sampling rates: EEG/ECG/EMG 256 Hz, movement 25 Hz.

In [ ]:
stem = runs[6]
sz_on, sz_off = seizeit2_events("sub-001")[stem][0]
sig = {}
for m in ["eeg", "ecg", "emg", "mov"]:
    path = stem.replace("/eeg/", f"/{m}/") + f"_{m}.edf"
    r = mne.io.read_raw_edf(fetch(SEIZEIT2 + path, "seizeit2/" + path), verbose="error")
    sig[m] = (r.get_data(tmin=sz_on - 120, tmax=sz_off + 120), r.info["sfreq"], r.ch_names)
    print(f"{m}: {r.ch_names} at {r.info['sfreq']} Hz, run length {r.n_times / r.info['sfreq'] / 3600:.1f} h")

In [ ]:
eeg = mne.filter.filter_data(sig["eeg"][0], 256, 0.5, 40, verbose="error") * 1e6
acc = np.linalg.norm(sig["mov"][0][:3], axis=0)  # magnitude of the accelerometer on the EEG device
fig, ax = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
for i, (y_, fs_, label) in enumerate([(eeg[0], 256, "EEG BTEleft [uV]"), (eeg[1], 256, "EEG CROSStop [uV]"),
                                      (sig["ecg"][0][0] * 1e6, 256, "ECG [uV]"), (acc, 25, "acceleration (a.u.)")]):
    ax[i].plot(np.arange(len(y_)) / fs_ - 120, y_, lw=0.4)
    ax[i].axvspan(0, sz_off - sz_on, color="red", alpha=0.15)
    ax[i].set_ylabel(label)
ax[-1].set_xlabel("time relative to seizure onset [s]"); plt.show()

Behind-the-ear EEG shows the seizure far less clearly than scalp EEG. The ECG, however, often reacts: many focal seizures
speed up the heart (*ictal tachycardia*). The function below counts R-peaks (the tall spikes of the ECG).

**Exercise 2.2.** Compute the heart rate (beats per minute) before the seizure (`-120 s` to `-20 s`), during the seizure
and after it (`+0` to `+120 s` after the end). Is ECG a useful second modality for a wearable detector?

In [ ]:
def heart_rate(ecg, fs):
    """Beats per minute from an ECG segment (simple R-peak detector)."""
    x = mne.filter.filter_data(ecg[None], fs, 5, 30, verbose="error")[0]
    x = x if x.max() > -x.min() else -x  # R-peaks may point down, depending on the electrode position
    peaks, _ = find_peaks(x, distance=int(0.3 * fs), height=0.5 * np.percentile(x, 98))
    return 60 * len(peaks) / (len(x) / fs)

ecg = sig["ecg"][0][0]
d = int((sz_off - sz_on) * 256)  # seizure length in samples; the segment starts 120 s before onset
# TODO: hr_before, hr_during, hr_after
pass  # YOUR CODE HERE
print(f"heart rate: before {hr_before:.0f}, during {hr_during:.0f}, after {hr_after:.0f} bpm")
assert hr_during > hr_before

*Your answer:*

## Part 3 — Preprocessing: every step has a reason
Back to CHB-MIT (`raw` of `chb01_03`). The pipeline, in order:

| Step | What | Why |
|---|---|---|
| channel selection | 18 double-banana channels | same input for every file, no duplicates |
| band-pass 0.5–40 Hz | remove slow drifts and everything above 40 Hz | drifts (sweat, movement) and line noise / muscle artifacts are not brain activity of interest |
| resample to 128 Hz | halve the sampling rate | after a 40-Hz low-pass, 128 Hz is enough (Nyquist: 64 Hz > 40 Hz); half the data |
| windowing | 4-s windows | models need fixed-size inputs |
| labelling | window = seizure if … | defines what the model learns |
| artifact check | flat / extreme windows | broken electrodes produce garbage |
| normalisation | mean / std from the **training** set | comparable scales, no test information |

A **notch filter** (50 or 60 Hz) is only needed if you keep frequencies above the line frequency; our 40-Hz low-pass already
removes it. Note: MNE's filters are *zero-phase*, i.e. they use future samples. Fine for offline analysis — a real-time
detector needs causal filters (day 6).

**Exercise 3.1.** Apply the course function `preprocess(raw.copy(), CHANNELS, 128)` (pick + band-pass 0.5–40 Hz + resample)
and store the result in `data` (shape: channels × time, in µV). Then compute the PSD of channel `F7-T7` (index 1) of the
background segment `onset-300 … onset-260 s` (careful: now 128 samples per second!) and check that the 60-Hz peak is gone.

In [ ]:
# TODO: data, f_f, p_f (PSD of the filtered background segment)
pass  # YOUR CODE HERE
print("data:", data.shape, "| power near 60 Hz no longer exists: highest frequency =", f_f.max(), "Hz")
assert data.shape == (18, 3600 * 128) and f_f.max() == 64

**Exercise 3.2 — why resample?** Compute the memory (in MB, float32 = 4 bytes) of one hour of 18-channel EEG at 256 Hz
(`mb_256`) and at 128 Hz (`mb_128`), and the memory of all 40.6 h of chb01 at 128 Hz in GB (`gb_chb01`).

In [ ]:
# TODO: mb_256, mb_128, gb_chb01
pass  # YOUR CODE HERE
print(f"1 h: {mb_256:.0f} MB at 256 Hz, {mb_128:.0f} MB at 128 Hz | chb01 at 128 Hz: {gb_chb01:.2f} GB")
assert round(mb_128) == 33

### 3.3 Windows and labels
Models get fixed-size inputs: we cut the recording into **windows** of `win_s` seconds that start every `step_s` seconds.
`step_s < win_s` gives **overlapping** windows — more training examples (especially of the rare seizures), but neighbouring
windows are almost identical.

**Labelling rule.** Which windows count as "seizure"? Options: the window centre lies in the seizure (the rule of the course
module), *any* overlap, or at least 50 % overlap. Seizure onsets are themselves uncertain by a few seconds (experts disagree),
so the rule matters mostly at the edges — but it must be fixed and reported.

**Exercise 3.3.** Implement `sliding_windows(data, fs, win_s, step_s)` → `X` of shape (n, channels, win_s·fs) and the start
times `t0` (seconds), and `label_windows(t0, win_s, events, min_overlap)` → `y` (1 if at least `min_overlap` of the window lies
inside a seizure).

In [ ]:
def sliding_windows(data, fs, win_s, step_s):
    n, step = int(win_s * fs), int(step_s * fs)
    starts = np.arange(0, data.shape[1] - n + 1, step)
    # TODO: X (n_windows, channels, n) and t0 (start time of every window in seconds)
    pass  # YOUR CODE HERE
    return X, t0

def label_windows(t0, win_s, events, min_overlap=0.5):
    y = np.zeros(len(t0), dtype=np.int64)
    # TODO: overlap of every window with every seizure (a, b), as a fraction of win_s
    pass  # YOUR CODE HERE
    return y

Xo, t0 = sliding_windows(data, 128, 4.0, 1.0)
yo = label_windows(t0, 4.0, [(onset, end)])
Xn, yn = windows(data, 128, 4.0, [(onset, end)])  # course module: non-overlapping, centre rule
print(f"4-s windows, 1-s step: {len(yo)} windows, {yo.sum()} seizure | non-overlapping: {len(yn)} windows, {yn.sum()} seizure")
assert Xo.shape == (3597, 18, 512) and yo.sum() == 41 and yn.sum() == 10

*Your answer:*

### 3.4 Artifact checks
Broken or loose electrodes give **flat** channels (std ≈ 0) or **extreme** jumps. A common recipe is to drop windows whose
peak-to-peak amplitude exceeds a threshold. But: seizures *are* high-amplitude activity…

**Exercise 3.4.** For the non-overlapping windows `Xn, yn`, compute the peak-to-peak amplitude per window and channel
(`ptp`, shape (n, 18)). Flag a window if *any* channel exceeds `thr` µV. Print the fraction of flagged background windows
and flagged seizure windows for `thr` = 300, 500, 800 µV, and count flat windows (any channel std < 1 µV) in `n_flat`.

In [ ]:
# TODO: ptp, the flagged fractions per threshold, n_flat
pass  # YOUR CODE HERE
print("flat windows:", n_flat)

*Your answer:*

## Part 4 — Splits and class imbalance
Now the full pipeline for six records of chb01 (`load_chbmit()` = download + the preprocessing above + 4-s windows).

In [ ]:
X, y, rec = load_chbmit()
records = ["chb01_03", "chb01_04", "chb01_15", "chb01_16", "chb01_18", "chb01_26"]
for i, r in enumerate(records):
    print(f"{r}: {np.sum(rec == i):4d} windows, {int(y[rec == i].sum()):3d} seizure windows")

**Why not a random split?** If we shuffle all windows and split them randomly, the test set contains windows whose direct
neighbours (one window earlier, same recording, same seizure) are in the training set. The model is tested on near-copies of
its training data and the score is too optimistic. The unit of generalisation decides the split: a **new recording** (or a
**new patient**, for cross-patient models) must not have been seen in training.

The cell below shows it: with a random split, for every seizure window in the test set the most similar training window
(nearest neighbour in band-power features) is a seizure window — half of them are the direct temporal neighbour.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

F = band_power(X)  # (n, 18 channels * 5 bands) log band power, from the course module
tr, te = train_test_split(np.arange(len(y)), test_size=0.3, random_state=0)
Fs = (F - F[tr].mean(0)) / F[tr].std(0)
sz_te = te[y[te] == 1]
nn_idx = tr[NearestNeighbors(n_neighbors=1).fit(Fs[tr]).kneighbors(Fs[sz_te], return_distance=False)[:, 0]]
print(f"random split: nearest training window is a seizure for {100 * y[nn_idx].mean():.0f} % of the test seizure windows, "
      f"the direct neighbour (same record, ±1 window) for {100 * np.mean(np.abs(nn_idx - sz_te) <= 1):.0f} %")

**Exercise 4.1.** Make a **record-wise** split: records 0–3 for training, records 4–5 (`chb01_18`, `chb01_26`) for testing.
Create the boolean masks `train` and `test`. Print the seizure fraction in both.

In [ ]:
# TODO: train, test (boolean masks over all windows)
pass  # YOUR CODE HERE
print(f"train: {train.sum()} windows, {100 * y[train].mean():.1f} % seizure | test: {test.sum()} windows, {100 * y[test].mean():.1f} % seizure")
assert not (train & test).any() and set(rec[train]).isdisjoint(set(rec[test]))

In a real project you also need a **validation** set (again whole records or patients) to choose hyper-parameters and the
decision threshold — the test set is used once, at the end (day 3).

## Part 5 — PyTorch `Dataset` and `DataLoader`
A `Dataset` answers two questions: how many samples (`__len__`) and what is sample `i` (`__getitem__`). A `DataLoader` turns it
into shuffled mini-batches. Normalisation belongs here: per-channel mean and std computed on the **training** windows only.

**Exercise 5.1.** Complete `EEGWindows`. `__getitem__` returns the standardised window as a float32 tensor of shape (18, 512)
and the label as a tensor. Compute `mean` and `std` per channel (shape (18, 1)) from the training windows.

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class EEGWindows(Dataset):
    def __init__(self, X, y, mean, std):
        self.X, self.y, self.mean, self.std = X, y, mean, std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        # TODO: return (standardised window as float32 tensor, label as tensor)
        pass  # YOUR CODE HERE

# TODO: mean, std per channel from the training windows, shape (18, 1)
pass  # YOUR CODE HERE
train_ds = EEGWindows(X[train], y[train], mean, std)
test_ds = EEGWindows(X[test], y[test], mean, std)  # same mean/std as training!
xb, yb = next(iter(DataLoader(train_ds, batch_size=64, shuffle=True)))
print("batch:", tuple(xb.shape), tuple(yb.shape), "| per-channel mean of a batch:", xb.mean((0, 2))[:4])
assert xb.shape == (64, 18, 512) and xb.dtype == torch.float32

**Class imbalance.** Only ~1 % of the training windows are seizures: most batches contain none. A
`WeightedRandomSampler` draws samples with probabilities that you choose — e.g. inversely proportional to the class size, so
that both classes appear equally often (oversampling the seizures). Alternatives: class weights in the loss (day 3).

**Exercise 5.2.** Give every training window the weight `1 / (number of training windows of its class)` and build a
`DataLoader` with `WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)`. Measure the average fraction of
seizure windows per batch (`frac`).

In [ ]:
# TODO: weights, loader, frac
pass  # YOUR CODE HERE
print(f"seizure fraction per batch: {frac:.2f} (without sampler: {y[train].mean():.3f})")
assert 0.35 < frac < 0.65

*Your answer:*

## Part 6 — A sanity baseline and the right metrics
Before any deep model: a **simple baseline** tells you whether the pipeline works and what a deep network has to beat. We use
the log band power per channel (delta, theta, alpha, beta, low gamma) and logistic regression with balanced class weights.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
clf.fit(F[train], y[train])  # the scaler is fitted on training data only, too
score = clf.predict_proba(F[test])[:, 1]

**Metrics for seizure detection** (window level, 4-s windows):
- **sensitivity** = detected seizure windows / all seizure windows (missed seizures are dangerous)
- **specificity** = correctly rejected background windows / all background windows
- **false alarms per hour (FA/h)** = false-positive windows / hours of test data (what patients and nurses experience)
- **AUPRC** = area under the precision–recall curve, threshold-free, meaningful under imbalance (chance level = seizure fraction)

**Exercise 6.1.** Implement `detection_metrics(y_true, score, thr=0.5, win_s=4)` returning a dict with `accuracy`,
`sensitivity`, `specificity`, `fa_per_h` and `auprc`. Evaluate the baseline and a "model" that always says no seizure
(`score = np.zeros(...)`).

In [ ]:
def detection_metrics(y_true, score, thr=0.5, win_s=4.0):
    pred = score >= thr
    # TODO: accuracy, sensitivity, specificity, fa_per_h, auprc
    pass  # YOUR CODE HERE

for name, s in [("logistic regression", score), ("always 'no seizure'", np.zeros(test.sum()))]:
    m = detection_metrics(y[test], s)
    print(f"{name:20s} " + "  ".join(f"{k} {v:.3f}" for k, v in m.items()))
m = detection_metrics(y[test], score)
assert 0.9 < m["accuracy"] < 1 and m["sensitivity"] > 0.5

**Exercise 6.2 — choose the threshold.** Print sensitivity and FA/h for thresholds 0.5, 0.8, 0.9, 0.95, 0.99. A wearable
alarm should give at most ~1 false alarm per hour (better: per day). Which threshold would you choose, and what does it cost?

In [ ]:
# TODO: sensitivity and FA/h per threshold
pass  # YOUR CODE HERE

*Your answer:*

## Wrap-up — the pipeline you built
`EDF files` → channel selection (18 bipolar) → band-pass 0.5–40 Hz → resample 128 Hz → 4-s windows + labelling rule →
artifact check (against the seizure windows!) → **record-/patient-wise split** → normalisation with training statistics →
`Dataset` / `DataLoader` (with sampler for imbalance) → baseline → sensitivity, FA/h, AUPRC.

Every arrow is a decision that you will have to make — and report — in your project. Most "great" results in the seizure
detection literature that do not hold up fail at one of these arrows (split, labelling, metric), not at the model.